Starting from the beginning, the first thing that we need to do is read in the data from output.csv

In [1]:
import pandas as pd

In [2]:
deepmind_data = pd.read_csv('/home/gridsan/tmackey/materials_discovery/data/gnome_data/output.csv')

In [3]:
#let's check the shape of deepmind_data
deepmind_data.shape

(377223, 2)

It has 377223 columns as expected

In [4]:
deepmind_data

,composition,cif
0,by_composition/,NaN
1,by_composition/B24Dy6Ru4Yb2.CIF,# generated using pymatgen\ndata_YbDy3(B6Ru)2\...
2,by_composition/Br2Fe4Ho1Tm3.CIF,# generated using pymatgen\ndata_HoTm3(Fe2Br)2...
3,by_composition/C5Nb4Tm2.CIF,# generated using pymatgen\ndata_Tm2Nb4C5\n_sy...
4,by_composition/Os1S6Ta2.CIF,# generated using pymatgen\ndata_Ta2OsS6\n_sym...
...,...,...
377218,by_composition/Fe21Tm6U2.CIF,# generated using pymatgen\ndata_Tm6U2Fe21\n_s...
377219,by_composition/Cl2Ge2La4Rh2.CIF,# generated using pymatgen\ndata_La2GeRhCl\n_s...
377220,by_composition/Dy1Er11Os3Tc1.CIF,# generated using pymatgen\ndata_DyEr11TcOs3\n...
377221,by_composition/Br12Ce1Rb4Re1.CIF,# generated using pymatgen\ndata_Rb4CeReBr12\n...


It has a composition and a cif column, which is nice. 

In [14]:
#now, let's split the data into train, test and val sets. We can use the torch split function for this

# Shuffle the DataFrame
deepmind_data = deepmind_data.sample(frac=1).reset_index(drop=True)

In [15]:
# Split the data
dm_train = deepmind_data.sample(frac=0.6, random_state=1) # 60% for training

In [16]:
dm_remaining = deepmind_data.drop(dm_train.index)

In [17]:
dm_val = dm_remaining.sample(frac=0.5, random_state=1) # 20% for validation
dm_test = dm_remaining.drop(dm_val.index) # Remaining 20% for testing

Let's check shapes of the dataframes.

In [18]:
print("train shape is", dm_train.shape) 
print("val shape is", dm_val.shape)
print("test shape is", dm_test.shape)

train shape is (226334, 2)
val shape is (75444, 2)
test shape is (75445, 2)


In [19]:
(226334 + 75444 + 75445) == len(deepmind_data)

True

Looks good! Let's go ahead and save these to train, test, and val files in an appropriate folder before we move on with further processing.

In [20]:
dm_train.to_csv('/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/train.csv', index=False)
dm_val.to_csv('/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/val.csv', index=False)
dm_test.to_csv('/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/test.csv', index=False)

The next step in the pipline is getting xrd data for all of the entries. Because this is fairly computationally intensive, we'll do this in a distributed way. 

In [8]:
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator

import pandas as pd

import sys
import os
import numpy as np
#read in the worker number 
try: 
    worker_num = int(sys.argv[1])
except: 
    worker_num = 0

num_splits = 100

print("worker_num", worker_num)
print("num_splits", num_splits)


data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

#load in the data 
train_df = pd.read_csv(data_dir + 'train.csv')
test_df = pd.read_csv(data_dir + 'test.csv')
val_df = pd.read_csv(data_dir + 'val.csv')

# Initialize the XRDCalculator with a wavelength of CuKa (1.54060 Å)
xrd_calculator = XRDCalculator(wavelength='CuKa')
from tqdm.auto import tqdm
tqdm.pandas()

def get_xrd_information(crystal_str):
    try: 
        crystal = Structure.from_str(crystal_str, fmt='cif')
    except:
        crystal = None

    try:  
        xrd = xrd_calculator.get_pattern(crystal)
    except: 
        xrd = None

    try: 
        x = xrd.x.tolist()
        y = xrd.y.tolist()
    except:
        x = None
        y = None

    try: 
        atomic_species = [Element(specie).Z for specie in crystal.species]
    except: 
        atomic_species = None

    return [xrd, x, y, atomic_species]

data_frames = {"train": train_df, "test": test_df, "val": val_df}

for name, df in data_frames.items():
    
    chunk_size = np.ceil(len(df)/num_splits)
    
    start_index = int(worker_num*chunk_size)
    end_index = int(min(start_index + chunk_size, len(df))) #prevents end index > len(df)
    
    print("start_index", start_index)
    print("end_index", end_index)

    sub_df = df.iloc[start_index:end_index].copy()
    sub_crystals = sub_df['cif'].progress_apply(get_xrd_information)
    sub_df['xrd'] = sub_crystals.progress_apply(lambda x: x[0])
    sub_df['xrd_peak_locations'] = sub_crystals.progress_apply(lambda x: x[1])
    sub_df['xrd_peak_intensities'] = sub_crystals.progress_apply(lambda x: x[2])
    sub_df['atomic_numbers'] = sub_crystals.progress_apply(lambda x: x[3])

    #save the csv
    sub_df.to_csv(data_dir + f'{name}_xrd_{worker_num}.csv', index=False)

Now that we have the data created, we just need to merge it together 

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

#assuming all of the data was stored as .pt dictionaries of string keys and graph values
num_workers = 100
dataset_names =  ['train', 'test', 'val']
data_source = "mp_20_dm"
data_dir = f"/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/"

for name in dataset_names:
    total_df = pd.DataFrame()
    for worker_num in tqdm(range(num_workers)):
        df = pd.read_csv(data_dir + f'{name}_xrd_{worker_num}.csv')
        #add df without using append
        total_df = pd.concat([total_df, df])
    total_df.to_csv(f"/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/{name}_xrd.csv", index=False)
    print(f"Saved {name} dataset")

100%|██████████| 100/100 [02:02<00:00,  1.23s/it]


Saved train dataset


100%|██████████| 100/100 [00:46<00:00,  2.17it/s]


Saved test dataset


100%|██████████| 100/100 [00:46<00:00,  2.17it/s]


Saved val dataset


Now, let's read the data back in and assess the quality 

Now, we need to get rid of empty rows. Because the data is now so big, it's best to do this one at a time. Let's do the test data first because it's relatively small.

In [3]:
import pandas as pd
import numpy as np 

In [4]:
test_df = pd.read_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/test_xrd.csv")

In [5]:
print("test_df was ", test_df.shape)

test_df was  (75445, 6)


In [6]:
#check to see if the xrd_peak_locations has any null values
# returns True if there are null values
print("are there nulls in the test xrd_peak_locations? ", test_df['xrd_peak_locations'].isnull().values.any())
print("are there nulls in the test xrd_peak_intensities? ", test_df['xrd_peak_intensities'].isnull().values.any())
print("are there nulls in the test atomic_numbers? ", test_df['atomic_numbers'].isnull().values.any())


are there nulls in the test xrd_peak_locations?  True
are there nulls in the test xrd_peak_intensities?  True
are there nulls in the test atomic_numbers?  True


In [7]:
#find the location of the nulls in each column
null_locations = test_df['xrd_peak_locations'].isnull()
print("indices of nulls in the xrd peak locations: ", np.where(null_locations))

null_locations = test_df['xrd_peak_intensities'].isnull()
print("indices of nulls in the xrd peak intensities: ", np.where(null_locations))

null_locations = test_df['atomic_numbers'].isnull()
print("indices of nulls in the atomic numbers: ", np.where(null_locations))

indices of nulls in the xrd peak locations:  (array([73585]),)
indices of nulls in the xrd peak intensities:  (array([73585]),)
indices of nulls in the atomic numbers:  (array([73585]),)


In [8]:
#remove the null values from the dataframe
test_df = test_df.dropna()

In [9]:
print("test_df is now ", test_df.shape)

test_df is now  (75444, 6)


In [10]:
# check to see if the null values were removed
# returns True if there are null values
print("are there nulls in the test xrd_peak_locations? ", test_df['xrd_peak_locations'].isnull().values.any())
print("are there nulls in the test xrd_peak_intensities? ", test_df['xrd_peak_intensities'].isnull().values.any())
print("are there nulls in the test atomic_numbers? ", test_df['atomic_numbers'].isnull().values.any())

are there nulls in the test xrd_peak_locations?  False
are there nulls in the test xrd_peak_intensities?  False
are there nulls in the test atomic_numbers?  False


In [11]:
#save the dataframe as a csv
test_df.to_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/test_xrd.csv", index=False)

Repeat for the val data

In [12]:
import pandas as pd
import numpy as np

In [13]:
#repeat the process with the val data
val_df = pd.read_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/val_xrd.csv")

#check to see if the xrd_peak_locations has any null values
# returns True if there are null values
print("are there nulls in the val xrd_peak_locations? ", val_df['xrd_peak_locations'].isnull().values.any())
print("are there nulls in the val xrd_peak_intensities? ", val_df['xrd_peak_intensities'].isnull().values.any())
print("are there nulls in the val atomic_numbers? ", val_df['atomic_numbers'].isnull().values.any())

print("val_df is now ", val_df.shape)

#find the location of the nulls in each column
null_locations = val_df['xrd_peak_locations'].isnull()
print("indices of nulls in the xrd peak locations: ", np.where(null_locations))

null_locations = val_df['xrd_peak_intensities'].isnull()
print("indices of nulls in the xrd peak intensities: ", np.where(null_locations))

null_locations = val_df['atomic_numbers'].isnull()
print("indices of nulls in the atomic numbers: ", np.where(null_locations))

#remove the null values from the dataframe
val_df = val_df.dropna()

print("val_df is now ", val_df.shape)

# check to see if the null values were removed
# returns True if there are null values
print("are there nulls in the val xrd_peak_locations? ", val_df['xrd_peak_locations'].isnull().values.any())
print("are there nulls in the val xrd_peak_intensities? ", val_df['xrd_peak_intensities'].isnull().values.any())
print("are there nulls in the val atomic_numbers? ", val_df['atomic_numbers'].isnull().values.any())

#save the dataframe as a csv
val_df.to_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/val_xrd.csv", index=False)

are there nulls in the val xrd_peak_locations?  False
are there nulls in the val xrd_peak_intensities?  False
are there nulls in the val atomic_numbers?  False
val_df is now  (75444, 6)
indices of nulls in the xrd peak locations:  (array([], dtype=int64),)
indices of nulls in the xrd peak intensities:  (array([], dtype=int64),)
indices of nulls in the atomic numbers:  (array([], dtype=int64),)
val_df is now  (75444, 6)
are there nulls in the val xrd_peak_locations?  False
are there nulls in the val xrd_peak_intensities?  False
are there nulls in the val atomic_numbers?  False


Repeat for the train data

In [1]:
import pandas as pd
import numpy as np

In [2]:
#repeat the process with the val data
train_df = pd.read_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/train_xrd.csv")

#check to see if the xrd_peak_locations has any null values
# returns True if there are null values
print("are there nulls in the train xrd_peak_locations? ", train_df['xrd_peak_locations'].isnull().values.any())
print("are there nulls in the train xrd_peak_intensities? ", train_df['xrd_peak_intensities'].isnull().values.any())
print("are there nulls in the train atomic_numbers? ", train_df['atomic_numbers'].isnull().values.any())

print("train_df is now ", train_df.shape)

#find the location of the nulls in each column
null_locations = train_df['xrd_peak_locations'].isnull()
print("indices of nulls in the xrd peak locations: ", np.where(null_locations))

null_locations = train_df['xrd_peak_intensities'].isnull()
print("indices of nulls in the xrd peak intensities: ", np.where(null_locations))

null_locations = train_df['atomic_numbers'].isnull()
print("indices of nulls in the atomic numbers: ", np.where(null_locations))

#remove the null values from the dataframe
train_df = train_df.dropna()

print("train_df is now ", train_df.shape)


# check to see if the null values were removed
# returns True if there are null values
print("are there nulls in the train xrd_peak_locations? ", train_df['xrd_peak_locations'].isnull().values.any())
print("are there nulls in the train xrd_peak_intensities? ", train_df['xrd_peak_intensities'].isnull().values.any())
print("are there nulls in the train atomic_numbers? ", train_df['atomic_numbers'].isnull().values.any())


#save the dataframe as a csv
train_df.to_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/train_xrd.csv", index=False)

are there nulls in the train xrd_peak_locations?  True
are there nulls in the train xrd_peak_intensities?  True
are there nulls in the train atomic_numbers?  True
train_df is now  (226334, 6)
indices of nulls in the xrd peak locations:  (array([22040]),)
indices of nulls in the xrd peak intensities:  (array([22040]),)
indices of nulls in the atomic numbers:  (array([22040]),)
train_df is now  (226333, 6)
are there nulls in the train xrd_peak_locations?  False
are there nulls in the train xrd_peak_intensities?  False
are there nulls in the train atomic_numbers?  False


Now, we need to get the disc sim xrd data

In [3]:
import pandas as pd
import numpy as np 
import ast

from tqdm.auto import tqdm
tqdm.pandas()

In [4]:
#start with the test df
test_df = pd.read_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/test_xrd.csv")
df = test_df

In [5]:
#let's pull out the diffraction patterns ahead of time 
def simulate_xrd(peak_locations, peak_intensities, lower_bound = 5, upper_bound = 75, dimensions = 200):
    interval =  (upper_bound - lower_bound)/dimensions
    sim_positions = np.arange(lower_bound, upper_bound, interval)
    # Create an empty intensity array for the simulation
    sim_intensities = np.zeros_like(sim_positions)
    
    # Loop over all simulated positions
    for i, pos in enumerate(sim_positions):
        # Find peak locations within 0.25° of the current simulated position
        close_peaks = [(loc, intensity) for loc, intensity in zip(peak_locations, peak_intensities) if abs(loc - pos) <= interval/2]
        
        # If there are close peaks, sum the intensities among those peaks
        if close_peaks:
            intensities = np.array([intensity for loc, intensity in close_peaks])
            sim_intensities[i] = np.sum(intensities)
    
    sim_intensities = 100*sim_intensities / max(sim_intensities)
    
    return sim_intensities

In [6]:
df['xrd_peak_locations'].iloc[0]

'[12.906496391514562, 12.90676542623153, 18.291171298165246, 18.291938516688404, 22.45066883219385, 22.451141284075575, 25.980459307060915, 25.98100800962597, 29.110441873881186, 29.11143193271356, 29.111554852064977, 31.960507173535298, 34.598298550895805, 34.59882836707439, 34.599145851884735, 37.070404979539504, 37.07200279402078, 39.40932795361331, 39.409991406764256, 39.410749554009485, 39.41084328286555, 41.63735671160709, 41.638444122963485, 41.63880507485728, 43.77248443130054, 43.77291905722523, 45.826399329871265, 45.82740555352414, 47.81095176681979, 47.81168189200389, 47.81225043404681, 47.81289924672073, 47.81298218670159, 51.60724833512571, 51.60886074382036, 51.609090480105316, 53.432171435342816, 53.43336855680522, 55.21362506766378, 55.21428347113773, 55.21479590619587, 55.215455556309095, 56.95868335137557, 56.959544079351744, 56.95983483791742, 58.66778368103775, 58.6685574057514, 58.66968425962487, 58.670036012180894, 58.670249486371816, 60.34776835207664, 60.349985

Unfortunately, this will also take foreever, so we'll need to do it in a distributed way. We can follow the same approach used for getting the xrd data.

In [2]:
train_df = pd.read_csv(data_dir + 'train_xrd.csv') 

In [12]:
import pandas as pd

import sys
import os
import numpy as np
import ast
from tqdm.auto import tqdm
tqdm.pandas()


#read in the worker number 
try: 
    worker_num = int(sys.argv[1])
except: 
    worker_num = 0

num_splits = 100
data_dir = '/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/'

#load in the data 
train_df = pd.read_csv(data_dir + 'train_xrd.csv') 
test_df = pd.read_csv(data_dir + 'test_xrd.csv')
#train and test commented out for testing purposes
val_df = pd.read_csv(data_dir + 'val_xrd.csv')
#let's pull out the diffraction patterns ahead of time 
def simulate_xrd(peak_locations, peak_intensities, lower_bound = 5, upper_bound = 75, dimensions = 200):
    interval =  (upper_bound - lower_bound)/dimensions
    sim_positions = np.arange(lower_bound, upper_bound, interval)
    # Create an empty intensity array for the simulation
    sim_intensities = np.zeros_like(sim_positions)
    
    # Loop over all simulated positions
    for i, pos in enumerate(sim_positions):
        # Find peak locations within 0.25° of the current simulated position
        close_peaks = [(loc, intensity) for loc, intensity in zip(peak_locations, peak_intensities) if abs(loc - pos) <= interval/2]
        
        # If there are close peaks, sum the intensities among those peaks
        if close_peaks:
            intensities = np.array([intensity for loc, intensity in close_peaks])
            sim_intensities[i] = np.sum(intensities)
    
    sim_intensities = 100*sim_intensities / max(sim_intensities)
    
    return sim_intensities

data_frames = {"train": train_df, "test": test_df, "val": val_df}

for name, df in data_frames.items():
    
    chunk_size = np.ceil(len(df)/num_splits)
    
    start_index = int(worker_num*chunk_size)
    end_index = int(min(start_index + chunk_size, len(df))) #prevents end index > len(df)
    sub_df = df.iloc[start_index:end_index].copy()

    sub_df['xrd_peak_locations'] = sub_df['xrd_peak_locations'].progress_apply(ast.literal_eval)
    sub_df['xrd_peak_intensities'] = sub_df['xrd_peak_intensities'].progress_apply(ast.literal_eval)
    sub_df['disc_sim_xrd'] = sub_df.progress_apply(lambda row: simulate_xrd(row['xrd_peak_locations'], row['xrd_peak_intensities']), axis=1)    

    #save
    sub_df.to_csv(data_dir + f'{name}_xrd_disc_sim_{worker_num}.csv', index=False)

After the sub csvs have been created, we can merge them together

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

#assuming all of the data was stored as .pt dictionaries of string keys and graph values
num_workers = 100
dataset_names =  ['train', 'test', 'val']
data_source = "mp_20_dm"
data_dir = f"/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/"

for name in dataset_names:
    total_df = pd.DataFrame()
    for worker_num in tqdm(range(num_workers)):
        df = pd.read_csv(data_dir + f'{name}_xrd_disc_sim_{worker_num}.csv')
        # print(len(df))
        #add df without using append
        total_df = pd.concat([total_df, df])
    total_df.to_csv(f"/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/{name}_xrd_disc_sim.csv", index=False)
    print(f"Saved {name} dataset")

100%|██████████| 100/100 [01:41<00:00,  1.01s/it]


Saved train dataset


100%|██████████| 100/100 [00:34<00:00,  2.90it/s]


Saved test dataset


100%|██████████| 100/100 [00:33<00:00,  2.97it/s]


Saved val dataset


Check for duplicates in the data. All of the data were calculated the same way, so we can just check one. Let's check train. 

In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv("/home/gridsan/tmackey/materials_discovery/data/actually_full_gnome_data/train_xrd_disc_sim.csv")

In [3]:
train_df.shape

(226333, 7)

In [4]:
#look for duplicates in the xrd peak locations
train_df['xrd_peak_intensities'].duplicated().sum()

0